# Rowan Qwen 1.7B Demo Training

This notebook trains two LoRA adapters for the `Love at Dusk` Rowan demo:

- Text generation adapter from `datasets/rowan_ashford_sft_all.jsonl`
- Reward/score adapter from `datasets/rowan_ashford_reward_all.jsonl`

Before running: set Colab to `Runtime -> Change runtime type -> GPU`.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Project Setup

Option A is easiest if this repo is on GitHub. Option B works if you upload a zip of the repo to Google Drive. Run only one option.


In [ ]:
# Option A: clone from GitHub. This repo does not include large local model folders.
REPO_URL = 'https://github.com/pianomaster99/isekai.git'

if REPO_URL:
    !rm -rf /content/isekai
    !git clone $REPO_URL /content/isekai
else:
    print('Set REPO_URL, or skip this cell and use Option B.')


In [ ]:
# Option B: unzip a repo archive from Drive. Upload isekai.zip to MyDrive first.
ZIP_PATH = '/content/drive/MyDrive/isekai.zip'

import os
if os.path.exists(ZIP_PATH):
    !rm -rf /content/isekai
    !unzip -q $ZIP_PATH -d /content
    print('Unzipped repo archive.')
else:
    print(f'No archive found at {ZIP_PATH}. Use Option A, or upload isekai.zip to Drive.')


In [ ]:
%cd /content/isekai
!pwd
!ls -la


## Install Dependencies

The local repo uses Hugging Face Transformers plus PEFT LoRA training.


In [ ]:
# Colab may preinstall an old torchao that conflicts with current PEFT LoRA injection.
!pip uninstall -y torchao
!pip install -U transformers peft accelerate safetensors


## Model Path

If `./qwen3-1.7b` is included in the repo/archive, leave this as-is. Otherwise set `BASE_MODEL` to a Hugging Face model id or a Drive path containing the downloaded Qwen model.


In [ ]:
import os, torch

LOCAL_MODEL = './qwen3-1.7b'
HF_MODEL = 'Qwen/Qwen3-1.7B'
BASE_MODEL = LOCAL_MODEL if os.path.exists(LOCAL_MODEL) else HF_MODEL

print('base model:', BASE_MODEL)
print('cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))


## Validate Datasets


In [ ]:
import json
from pathlib import Path

for path in ['datasets/rowan_ashford_sft_all.jsonl', 'datasets/rowan_ashford_reward_all.jsonl']:
    rows = []
    with Path(path).open() as handle:
        for line in handle:
            if line.strip():
                rows.append(json.loads(line))
    print(path, len(rows), 'rows')


## Train Text Generation Adapter


In [ ]:
!python train_text_model.py \
  --model-path $BASE_MODEL \
  --train-file datasets/rowan_ashford_sft_all.jsonl \
  --output-dir models/rowan-qwen3-1.7b-sft \
  --epochs 3 \
  --batch-size 1 \
  --grad-accum 8


## Train Reward Adapter


In [ ]:
!python train_reward_model.py \
  --model-path $BASE_MODEL \
  --train-file datasets/rowan_ashford_reward_all.jsonl \
  --output-dir models/rowan-qwen3-1.7b-reward \
  --epochs 5 \
  --batch-size 1 \
  --grad-accum 8


## Save Adapters To Drive


In [ ]:
!mkdir -p /content/drive/MyDrive/isekai-rowan-models
!cp -r models/rowan-qwen3-1.7b-sft /content/drive/MyDrive/isekai-rowan-models/
!cp -r models/rowan-qwen3-1.7b-reward /content/drive/MyDrive/isekai-rowan-models/
!tar -czf /content/drive/MyDrive/isekai-rowan-models/rowan-qwen3-1.7b-adapters.tar.gz models/rowan-qwen3-1.7b-sft models/rowan-qwen3-1.7b-reward
!ls -lh /content/drive/MyDrive/isekai-rowan-models


## Push Trained Adapters To GitHub

This uses Git LFS because adapter weights can exceed normal GitHub file limits. In Colab, add a secret named `GH_TOKEN` with a GitHub personal access token that has repo write access.


In [ ]:
from google.colab import userdata
import getpass
import os
from pathlib import Path

GITHUB_REPO = 'pianomaster99/isekai'
MODEL_BRANCH = 'main'
REPO_DIR = '/content/isekai'
try:
    GH_TOKEN = userdata.get('GH_TOKEN')
except Exception:
    GH_TOKEN = None
if not GH_TOKEN:
    GH_TOKEN = getpass.getpass('GitHub token: ')
assert GH_TOKEN, 'GH_TOKEN is required to push trained adapters to GitHub'

!apt-get update -qq && apt-get install -y -qq git-lfs
!git lfs install

if not Path(REPO_DIR, '.git').exists():
    !rm -rf $REPO_DIR
    !git clone https://github.com/$GITHUB_REPO.git $REPO_DIR
%cd /content/isekai

!git config user.email 'colab-training-bot@example.com'
!git config user.name 'Colab Training Bot'
!git remote set-url origin https://$GH_TOKEN@github.com/$GITHUB_REPO.git
!git checkout $MODEL_BRANCH
!git pull origin $MODEL_BRANCH
!git lfs track 'models/rowan-qwen3-1.7b-sft/**'
!git lfs track 'models/rowan-qwen3-1.7b-reward/**'
!git lfs track '*.safetensors'
!git add .gitattributes
!git add -f models/rowan-qwen3-1.7b-sft models/rowan-qwen3-1.7b-reward
!git status --short
!git commit -m 'Add trained Rowan adapters' || echo 'No model changes to commit'
!git push origin $MODEL_BRANCH


## Quick Inference Smoke Test

This loads the text adapter and asks Rowan for one short reply. It also disables old Colab torchao during PEFT adapter loading.


In [ ]:
import importlib.metadata
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

def disable_incompatible_torchao():
    try:
        version = importlib.metadata.version('torchao')
    except importlib.metadata.PackageNotFoundError:
        return
    version_parts = tuple(int(part) for part in version.split('.')[:2] if part.isdigit())
    if version_parts >= (0, 16):
        return
    import peft.import_utils as peft_import_utils
    peft_import_utils.is_torchao_available = lambda: False
    try:
        import peft.tuners.lora.torchao as peft_lora_torchao
        peft_lora_torchao.is_torchao_available = lambda: False
    except Exception:
        pass
    print(f'Disabled incompatible torchao {version}; LoRA inference does not need torchao.')

disable_incompatible_torchao()
adapter_path = 'models/rowan-qwen3-1.7b-sft'
tokenizer = AutoTokenizer.from_pretrained(adapter_path, trust_remote_code=True)
base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, trust_remote_code=True)
model = PeftModel.from_pretrained(base_model, adapter_path).eval()
if torch.cuda.is_available():
    model = model.to('cuda')

messages = [
    {'role': 'system', 'content': 'You are Rowan Ashford at The Last Light. Reply briefly, guardedly, and in character.'},
    {'role': 'user', 'content': 'I noticed you kept the cracked mug for me again.'},
]
tokenized = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors='pt')
prompt_ids = tokenized['input_ids'] if isinstance(tokenized, dict) else getattr(tokenized, 'input_ids', tokenized)
if torch.cuda.is_available():
    prompt_ids = prompt_ids.to('cuda')
with torch.inference_mode():
    output = model.generate(prompt_ids, max_new_tokens=120, temperature=0.7, top_p=0.9, do_sample=True, pad_token_id=tokenizer.eos_token_id)
print(tokenizer.decode(output[0, prompt_ids.shape[-1]:], skip_special_tokens=True))


## Game API Usage

After copying adapters back into the project, the game can choose text and scorer models explicitly.


In [ ]:
from game_api import GameNpcEngine

engine = GameNpcEngine(
    text_model_path=BASE_MODEL,
    text_adapter_path='models/rowan-qwen3-1.7b-sft',
    scorer='trained',
    reward_base_model_path=BASE_MODEL,
    reward_model_path='models/rowan-qwen3-1.7b-reward',
)
engine.model_config()
